In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

df = pd.read_excel("./train_data.xlsx", index_col=0)

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

lasso = Lasso(alpha=0.0985013289259387)
lasso.fit(X, y)

coef = pd.Series(lasso.coef_, index=X.columns)
selected_features = coef[coef != 0]
removed_features = coef[coef == 0]

print(selected_features.sort_values(ascending=False))

selected_X = X[selected_features.index]
df_selected = pd.concat([selected_X, y], axis=1)
df_selected.to_csv("selected_features_data.csv")

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

data = pd.read_csv("./selected_features_data.csv", index_col=0)
X = data.iloc[:, :-1]
y = data.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_pred = lr_model.predict(X_test_scaled)
y_pred_proba = lr_model.predict_proba(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print(f"{accuracy:.4f}")

conf_matrix = confusion_matrix(y_test, y_pred)
print(conf_matrix)

class_report = classification_report(y_test, y_pred, target_names=['0', '1'])
print(class_report)

joblib.dump(scaler, "./scaler_std.joblib")
joblib.dump(lr_model, "./logistic_regression_model.joblib")

In [ ]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

data1 = pd.read_csv("./6UP0_all.csv", index_col=0)
data2 = pd.read_csv("./selected_features_data.csv", index_col=0)

loaded_scaler = joblib.load("./scaler_std.joblib")
loaded_lr_model = joblib.load("./logistic_regression_model.joblib")

cols = data2.columns
df = data1[cols]
df = df.drop("label", axis=1)

df_scaled = loaded_scaler.transform(df)
pred = loaded_lr_model.predict(df_scaled)
pred_prob = loaded_lr_model.predict_proba(df_scaled)

df["pred"] = pred
df["pred_prob_0"] = pred_prob[:, 0]
df["pred_prob_1"] = pred_prob[:, 1]

df.to_csv("./predict_all.csv")